# NEMCast — table inspection

One section per table. Each shows shape, dtypes, first 5 rows, and a numeric summary.

Run from `notebooks/`, so paths are relative (`../data/...`).

**The four tables:**

| File | What it is | Timing |
|---|---|---|
| `dispatchprice` | Actual cleared prices | After the interval |
| `dispatchregionsum` | Actual regional physical state | After the interval |
| `predispatch/` | AEMO's forecast prices, every vintage | Before the interval |
| `predispatch_regionsum/` | AEMO's forecast demand & renewables | Before the interval |

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 200)

INTERIM = Path('../data/interim')

def inspect(df, name):
    print(f'=== {name} ===')
    print(f'shape: {df.shape[0]:,} rows x {df.shape[1]} cols')
    if 'SETTLEMENTDATE' in df.columns:
        print(f'range: {df.SETTLEMENTDATE.min()}  ->  {df.SETTLEMENTDATE.max()}')
    elif 'DATETIME' in df.columns:
        print(f'range: {df.DATETIME.min()}  ->  {df.DATETIME.max()}')
    print()
    display(pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'nulls': df.isna().sum(),
        'distinct': df.nunique(),
    }))
    print('\nfirst 5 rows:')
    display(df.head())

---
## 1. DISPATCHPRICE — actual cleared prices

The target variable. `RRP` is the Regional Reference Price: the offer price of the marginal unit.
The other eight `*RRP` columns are FCAS prices, co-optimised in the same dispatch run.

In [ ]:
price = pd.read_parquet(INTERIM / 'dispatchprice.parquet')
inspect(price, 'DISPATCHPRICE')

In [ ]:
price.describe().T

---
## 2. DISPATCHREGIONSUM — actual regional state

The physical picture behind each price. **Most of this is dispatch output**, produced by the same
solve that set the price — so it's leakage if used contemporaneously. The exceptions are the
bid-derived and forecast columns (`AVAILABLEGENERATION`, `AVAILABLELOAD`, `UIGF`).

In [ ]:
regionsum = pd.read_parquet(INTERIM / 'dispatchregionsum.parquet')
inspect(regionsum, 'DISPATCHREGIONSUM')

In [ ]:
regionsum.describe().T

---
## 3. PREDISPATCH PRICE — AEMO's forecast prices

Partitioned by month. Reading one month here to keep memory flat.

**Two time columns, and the distinction is the whole point:**
- `PREDISPATCH_RUN_DATETIME` — when the forecast was made
- `DATETIME` — the interval being forecast

Every target interval appears in ~40 rows, one per run. This is what lets you filter to
"forecasts available at 12:30" without using information from the future.

In [ ]:
pdp_files = sorted((INTERIM / 'predispatch').glob('*.parquet'))
print(f'{len(pdp_files)} monthly files:')
for f in pdp_files:
    print(' ', f.name)

In [ ]:
pdp = pd.read_parquet(pdp_files[0])
inspect(pdp, 'PREDISPATCH PRICE (2023-01)')

In [ ]:
# The vintage structure: one interval, forecast many times.
one = pdp[(pdp.REGIONID == 'SA1') & (pdp.DATETIME == pdp.DATETIME.iloc[500])]
print(f'interval {pdp.DATETIME.iloc[500]} forecast {len(one)} times:\n')
display(one[['PREDISPATCH_RUN_DATETIME', 'DATETIME', 'RRP']]
        .sort_values('PREDISPATCH_RUN_DATETIME'))

---
## 4. PREDISPATCH REGIONSUM — AEMO's forecast demand & renewables

Same vintage structure. `SS_WIND_UIGF` and `SS_SOLAR_UIGF` split the intermittent forecast —
worth keeping separate, since solar is predictable and diurnal while wind is volatile and
can collapse at any hour.

In [ ]:
pdr_files = sorted((INTERIM / 'predispatch_regionsum').glob('*.parquet'))
pdr = pd.read_parquet(pdr_files[0])
inspect(pdr, 'PREDISPATCH REGIONSUM (2023-01)')

In [ ]:
pdr.describe().T

---
## Cross-check: do the two forecast tables align?

They should join 1:1 on run time, interval, and region.

In [ ]:
KEYS = ['PREDISPATCH_RUN_DATETIME', 'DATETIME', 'REGIONID']
j = pdp.merge(pdr, on=KEYS, how='inner', suffixes=('_price', '_sum'))
print(f'price:  {len(pdp):,}')
print(f'sum:    {len(pdr):,}')
print(f'joined: {len(j):,}')
j.head()

---
## Notes

Add your column-by-column findings here as you work through them.

### DISPATCHPRICE
- `RRP` — target. Floor -$1,000, cap ~$20,300 (rises annually).
- `PRICE_STATUS` — `FIRM` means settled and unrevisable.
- `INTERVENTION` — already filtered to 0.

### DISPATCHREGIONSUM
- `DEMANDFORECAST` — a **delta** (±13 MW), not a level. Expected change over the interval.
- `UIGF` vs `SEMISCHEDULE_CLEAREDMW` — the gap is curtailment, and it's **caused by** price.
- `TOTALDEMAND` — operational demand, net of rooftop solar.

### PREDISPATCH
- Resolution is 30-minute, not 5-minute.
- `RRP1`..`RRP8` are demand-scenario sensitivities, not the main forecast.
